# Reach-to-Grasp Preprocessing and Loader Validation for CTM

## 1. Purpose and Dataset Conventions

This notebook documents the preprocessing pipeline used to convert the reach-to-grasp dataset into a form suitable for Continuous Thought Machine experiments.

The goal is not only to produce a working loader, but also to keep the preprocessing assumptions explicit and reproducible.

The notebook therefore distinguishes between:

- information stored directly in the released dataset,
- behaviour reproduced from the original preprocessing source code,
- derived metadata introduced by the current loader,
- and later choices made specifically for CTM experiments.

The source CSV files are treated as immutable.

Corrections, class assignments, board-sensor mappings, movement boundaries, and diagnostic flags are stored only as derived information.

### 1.1 Dataset

The dataset contains recordings from five right-handed participants performing reach-to-grasp movements toward eight objects:

| Class | Object |
|---:|---|
| 0 | Stick |
| 1 | Ball |
| 2 | Screw |
| 3 | Box |
| 4 | Knife |
| 5 | Glass |
| 6 | Assembly |
| 7 | Disk |

The motion-capture glove contains 25 tracked landmarks with three coordinates each:

$$
25 \times 3 = 75
$$

kinematic dimensions.

For the final CTM representation, only the five fingertip landmarks are retained:

    SensorID_4
    SensorID_9
    SensorID_14
    SensorID_19
    SensorID_24

This gives:

$$
5 \times 3 = 15
$$

kinematic input channels.

The full 75-dimensional glove representation is still used where required to reproduce the original movement-boundary logic.

### 1.2 Experimental blocks

The recordings are organised into five blocks:

    B0, B1, B2, B3, B4

B0 contains the calibration or model-fitting trials.

B1-B4 contain the later experimental trials.

The class-label and board-sensor conventions differ between B0 and B1-B4, so these blocks cannot be treated identically during preprocessing.

### 1.3 Processed recording structure

Each processed trial CSV contains several signal groups.

The glove kinematics are stored as:

    SensorID_0_x
    SensorID_0_y
    SensorID_0_z
    ...
    SensorID_24_x
    SensorID_24_y
    SensorID_24_z

The object-board measurements are stored as eight channels:

    board_data_0
    board_data_1
    board_data_2
    board_data_3
    board_data_4
    board_data_5
    board_data_6
    board_data_7

Additional signals and metadata include:

    palm_data
    glove_timestamps
    board_timestamps
    palm_timestamps
    hand_active
    block
    trial_number
    rep_number
    attempt_number
    target_object
    trial_success

The palm signal is retained as a diagnostic signal, but it is not assumed to define the final movement boundary.

### 1.4 Relevant project paths

The current project uses:

    Project root:
    /Users/arthu/Desktop/CTM_Res-Pro

    Processed reach-to-grasp data:
    /Users/arthu/Desktop/CTM_Res-Pro/data/reach_to_grasp_dataset/data

    Loader:
    /Users/arthu/Desktop/CTM_Res-Pro/models/ctm/data_loader.py

    Preprocessing notebook:
    /Users/arthu/Desktop/CTM_Res-Pro/models/ctm/02_reach_to_grasp_preprocessing.ipynb

The preprocessing pipeline developed in this notebook will later provide fixed-format trajectories for the CTM experiments.

The current objective is therefore:

    reconstruct and validate preprocessing first
                |
                v
         build CTM samples
                |
                v
        run small CTM pilot
                |
                v
      run larger experiment grid

## 2. Inventory and Successful-Attempt Selection

Before applying any movement preprocessing, the loader first determines which physical CSV files correspond to retained experimental recordings.

### 2.1 Physical files versus logical recording slots

The processed dataset contains:

$$
976
$$

physical recording files.

A logical recording slot is defined by:

    participant
    block
    trial_number
    rep_number

Grouping the files by these identifiers produces:

$$
960
$$

logical recording slots.

The difference is caused by 16 slots that contain a failed attempt followed by a successful retry.

In every one of these retry cases:

    attempt 0 -> trial_success = 0
    attempt 1 -> trial_success = 1

The loader therefore does not assume that `attempt0` is automatically the recording that should be retained.

### 2.2 Successful-attempt selection

For each logical recording slot, the loader selects the unique recording satisfying:

    trial_success == 1

The selection function also checks for ambiguous cases.

Its intended behaviour is:

    one successful attempt  -> retain it
    zero successful attempts -> exclude the slot
    multiple successful attempts -> raise an error

Across the complete dataset, there are 18 unsuccessful physical recordings.

Sixteen belong to retry slots and have successful replacements.

Two B0 slots have no successful replacement:

    participant 2, B0, trial 4, rep 0
    participant 3, B0, trial 3, rep 1

These two recordings are also explicitly marked unsuccessful in the original preprocessing source code.

The final retained dataset therefore contains:

$$
958
$$

successful logical recordings.

### 2.3 Retained dataset structure

The retained counts by participant are:

| Participant | Recordings |
|---:|---:|
| 1 | 192 |
| 2 | 191 |
| 3 | 191 |
| 4 | 192 |
| 5 | 192 |

The retained counts by block are:

| Block | Recordings |
|---:|---:|
| B0 | 318 |
| B1 | 160 |
| B2 | 160 |
| B3 | 160 |
| B4 | 160 |

This establishes the sample set used by all later preprocessing stages.

The important distinction is:

    976 physical files
          |
          v
    960 logical recording slots
          |
          v
    958 retained successful recordings

In [4]:
from pathlib import Path
import pandas as pd

from data_loader import (
    build_recording_inventory,
    extract_raw_metadata,
    select_successful_recordings,
)


# Build the physical-file inventory
inventory = build_recording_inventory()

# Extract raw metadata once for every physical recording
raw_records = []

for _, row in inventory.iterrows():
    raw_records.append(
        extract_raw_metadata(
            Path(row["path"])
        )
    )

raw_metadata_df = pd.DataFrame(raw_records)


# Select the unique successful recording for each logical slot
selected_df, excluded_df = select_successful_recordings(
    raw_metadata_df
)


# Count logical recording slots before success filtering
slot_columns = [
    "participant",
    "block",
    "trial_number",
    "rep_number",
]

n_logical_slots = (
    raw_metadata_df[
        slot_columns
    ]
    .drop_duplicates()
    .shape[0]
)


print("Physical recording files:", len(raw_metadata_df))
print("Logical recording slots:", n_logical_slots)
print("Retained successful recordings:", len(selected_df))
print("Excluded logical slots:", len(excluded_df))


print("\nExcluded slots:")
display(excluded_df)


print("\nRetained recordings per participant:")
display(
    selected_df[
        "participant"
    ]
    .value_counts()
    .sort_index()
    .rename("n_recordings")
    .to_frame()
)


print("\nRetained recordings per block:")
display(
    selected_df[
        "block"
    ]
    .value_counts()
    .sort_index()
    .rename("n_recordings")
    .to_frame()
)

Physical recording files: 976
Logical recording slots: 960
Retained successful recordings: 958
Excluded logical slots: 2

Excluded slots:


,participant,block,trial_number,rep_number,n_attempts,files
0,2,0,4,0,1,[B0_trial4_rep0_attempt0.csv]
1,3,0,3,1,1,[B0_trial3_rep1_attempt0.csv]



Retained recordings per participant:


,n_recordings
participant,
1,192
2,191
3,191
4,192
5,192



Retained recordings per block:


,n_recordings
block,
0,318
1,160
2,160
3,160
4,160


## 3. Target Correction and Class / Board Mapping

After successful recordings have been selected, the loader resolves the target metadata and derives the experimental class label and board-sensor channel.

These are treated as separate operations because `target_object`, `trial_number`, and physical board location do not have the same meaning in every block.

### 3.1 Final valid target value

The field `target_object` is not always constant within a recording.

Some processed CSV files contain more than one non-missing value, for example:

    [3, 0]

Across the 976 physical recordings, 139 files contain more than one distinct non-missing `target_object` value.

The loader therefore does not use the first row of `target_object`.

Instead, it records:

    target_object_final
    target_values_seen
    target_changed_within_recording

where `target_object_final` is the final valid, non-missing value in the recording.

The original evaluation code also uses the final target value when identifying the grasped object.

A previous diagnostic produced a count of 146 changing targets because transitions from missing values to populated values were also counted. The current count of 139 uses the stricter definition:

    more than one distinct non-missing target value

The two counts therefore describe different diagnostics rather than contradictory results.

### 3.2 Repetition-0 target correction

For B1-B4, repetition 0 follows an additional lookup rule provided by the lab supervisor.

For each combination of:

    participant
    block
    trial_number

the target for repetition 0 is obtained from the selected successful repetition-1 recording belonging to the same trial.

Conceptually:

    repetition 0
          |
          v
    find matching repetition 1
          |
          v
    use repetition 1 final valid target

This is a lookup operation, not an arithmetic transformation.

The loader therefore does not apply rules such as:

    target + 1

or:

    target - 1

The source CSV remains unchanged.

Instead, the derived metadata preserve both:

    target_object_raw
    target_object_corrected

together with information about whether the lookup rule was applied and which recording was used as the reference.

The correction applies to:

$$
5 \times 4 \times 8 = 160
$$

retained repetition-0 recordings in B1-B4.

In the current released dataset:

    correction lookups performed = 160
    numeric target values changed = 0

The rule is still kept explicitly because it records the intended experimental interpretation even when the stored final value already agrees with the reference repetition.

### 3.3 Block-specific class and board mapping

The experimental mapping differs between B0 and B1-B4.

For B0:

    class_label = trial_number
    board_column = board_data_3

For B1-B4:

    class_label = target_object_corrected
    board_column = board_data_{trial_number}

This means that object identity and physical board location must remain separate concepts.

For example, a B1-B4 recording with:

    trial_number = 1
    target_object_corrected = 6

is represented as:

    class_label = 6
    board_column = board_data_1

The loader therefore stores explicit provenance:

    class_label
    class_label_source
    board_sensor_index
    board_column
    board_sensor_source

### 3.4 Dataset-wide consistency

Applying this mapping to all 958 retained recordings produces:

| Class | Recordings |
|---:|---:|
| 0 | 120 |
| 1 | 120 |
| 2 | 120 |
| 3 | 119 |
| 4 | 119 |
| 5 | 120 |
| 6 | 120 |
| 7 | 120 |

The two missing samples are explained exactly by the two unreplaced B0 failures:

    participant 3, trial 3 -> class 3
    participant 2, trial 4 -> class 4

Each of B1-B4 contains exactly:

$$
20
$$

recordings per class.

The resulting balance provides an important validation that the reconstructed target, class, and board-sensor semantics are internally consistent.

In [5]:
from data_loader import (
    apply_rep0_target_correction,
    derive_experimental_mapping,
)


# Apply target correction and derive block-specific mapping
mapped_records = []

for _, row in selected_df.iterrows():
    metadata = row.to_dict()

    corrected = apply_rep0_target_correction(
        metadata,
        selected_df,
    )

    mapped = derive_experimental_mapping(
        corrected
    )

    mapped_records.append(mapped)

mapped_df = pd.DataFrame(mapped_records)


# ---------------------------------------------------------
# Correction validation
# ---------------------------------------------------------

rep0_corrected = mapped_df[
    mapped_df["target_correction_applied"]
]

print(
    "Correction lookups performed:",
    len(rep0_corrected),
)

print(
    "Numeric target values changed:",
    (
        rep0_corrected["target_object_raw"]
        != rep0_corrected["target_object_corrected"]
    ).sum(),
)


print(
    "\nCorrection lookups per block:"
)

display(
    rep0_corrected[
        "block"
    ]
    .value_counts()
    .sort_index()
    .rename("n_corrections")
    .to_frame()
)


# ---------------------------------------------------------
# Class-label validation
# ---------------------------------------------------------

print(
    "\nOverall class counts:"
)

display(
    mapped_df[
        "class_label"
    ]
    .value_counts()
    .sort_index()
    .rename("n_recordings")
    .to_frame()
)


print(
    "\nClass counts by block:"
)

display(
    pd.crosstab(
        mapped_df["block"],
        mapped_df["class_label"],
    )
)


# ---------------------------------------------------------
# Board-sensor validation
# ---------------------------------------------------------

print(
    "\nBoard columns by block:"
)

display(
    pd.crosstab(
        mapped_df["block"],
        mapped_df["board_column"],
    )
)


print(
    "\nClass-label source by block:"
)

display(
    pd.crosstab(
        mapped_df["block"],
        mapped_df["class_label_source"],
    )
)


print(
    "\nBoard-sensor source by block:"
)

display(
    pd.crosstab(
        mapped_df["block"],
        mapped_df["board_sensor_source"],
    )
)

Correction lookups performed: 160
Numeric target values changed: 0

Correction lookups per block:


,n_corrections
block,
1,40
2,40
3,40
4,40



Overall class counts:


,n_recordings
class_label,
0,120
1,120
2,120
3,119
4,119
5,120
6,120
7,120



Class counts by block:


...

board_column,board_data_0,board_data_1,board_data_2,board_data_3,board_data_4,board_data_5,board_data_6,board_data_7
block,,,,,,,,
0,0,0,0,318,0,0,0,0
1,20,20,20,20,20,20,20,20
2,20,20,20,20,20,20,20,20
3,20,20,20,20,20,20,20,20
4,20,20,20,20,20,20,20,20



Class-label source by block:


class_label_source,target_object_corrected,trial_number
block,,
0,0,318
1,160,0
2,160,0
3,160,0
4,160,0



Board-sensor source by block:


board_sensor_source,fixed_B0_sensor_3,trial_number
block,,
0,318,0
1,0,160
2,0,160
3,0,160
4,0,160


## 4. B0 Extraction Definition and Validation

The B0 movement interval is reconstructed directly from the original preprocessing code.

The implementation deliberately preserves several unusual details of the original helper functions so that reproduction and later methodological critique remain separate.

### 4.1 Preprocessing helpers used by B0

The threshold-crossing helper searches for the first sample satisfying:

$$
|x(t)| > \theta
$$

and returns one index after that first exceedance.

Therefore, if the first crossing occurs at Python index 2, the helper returns:

$$
3.
$$

This `+1` behaviour is preserved intentionally because it is part of the original implementation.

The glove-speed profile is calculated from all 75 glove coordinates:

$$
25 \times 3 = 75
$$

using the temporal gradient and Euclidean norm:

$$
v(t)
=
\left\|
\frac{d\mathbf{x}(t)}{dt}
\right\|_2.
$$

The full 75-dimensional glove signal is used only for reproducing the movement-boundary logic.

The later CTM representation will retain only the 15 fingertip coordinates.

### 4.2 Board-sensor cleanup

Before boundary detection, the original preprocessing replaces the board-sensor saturation values:

    8191
    8190

using the highest available value that is not equal to the target saturation value.

An important implementation detail is that this replacement is applied to the complete eight-channel board array.

Therefore, the replacement maximum can come from a different board channel than the channel later used for B0 boundary detection.

This behaviour is reproduced intentionally because it matches the original source code.

### 4.3 B0 endpoint

B0 always uses:

    board_data_3

as the board signal.

After saturation cleanup, the relative board signal is:

$$
b(t)
=
\text{board\_data\_3}(t)
-
\text{board\_data\_3}(0).
$$

The endpoint threshold is:

$$
\theta_{\mathrm{end}}
=
0.2 \max_t b(t).
$$

The detected endpoint is then obtained using the original threshold-crossing helper.

Because the helper applies an absolute value internally, the effective crossing condition is:

$$
|b(t)| > \theta_{\mathrm{end}}.
$$

This distinction is important for some of the flagged recordings discussed later.

### 4.4 B0 movement start

The start threshold is computed from the glove-speed profile before the detected endpoint:

$$
\theta_{\mathrm{start}}
=
0.1
\max_{t < t_{\mathrm{end}}}
v(t).
$$

The movement start is the first threshold crossing under the same helper rule.

Therefore, the B0 start used by the movement-export script is based on glove speed.

It is not defined by:

    hand_active

and it is not defined by:

    palm_data

Those signals remain useful diagnostics, but they represent different timing concepts.

### 4.5 Detected endpoint versus exported endpoint

The original movement-export script retains 50 samples after the detected endpoint:

    glove_data[start:end + 50]

The loader therefore preserves both:

    end_idx
    export_end_idx

with:

$$
\text{export\_end\_idx}
=
\text{end\_idx} + 50.
$$

This distinction is important for later CTM experiments.

The detected movement interval and the exported trajectory do not represent the same temporal extent.

For the median B0 recording:

    detected movement length = 81 samples
    exported trajectory length = 131 samples

The added 50 samples therefore account for approximately:

$$
\frac{50}{131} \approx 38\%
$$

of the exported sample count.

Consequently, a later window described as the "last quarter of the normalized export" should not automatically be interpreted as the last quarter of the physical reach.

Part of that window may occur after the detected endpoint.

### 4.6 Dataset-wide B0 validation

The reconstructed B0 extraction was applied to all 318 retained B0 recordings.

The result was:

| Check | Result |
|---|---:|
| Retained B0 recordings | 318 |
| Start/end pairs computed | 318 |
| Extraction exceptions | 0 |
| Export intervals exceeding recording length | 0 |

All 318 recordings produced ordered indices satisfying:

$$
t_{\mathrm{start}} < t_{\mathrm{end}}.
$$

This establishes computational reproducibility of the extraction rule.

It does not establish that every detected interval is physically plausible.

The detected movement lengths have:

    median = 81 samples
    middle 50% = 72-91 samples
    full range = 8-247 samples

The exported trajectories have:

    median = 131 samples
    full range = 58-297 samples

Physical plausibility remains under review for the flagged outliers described in the next section.

In [6]:
from pathlib import Path
import numpy as np
import pandas as pd

from data_loader import find_b0_movement_bounds


# Keep only the retained B0 recordings
b0_df = selected_df[
    selected_df["block"] == 0
].copy()


b0_results = []
b0_errors = []


for _, row in b0_df.iterrows():

    path = Path(row["path"])

    try:
        df = pd.read_csv(path)

        bounds = find_b0_movement_bounds(
            df
        )

        movement_length = (
            bounds["end_idx"]
            - bounds["start_idx"]
        )

        export_length = (
            bounds["export_end_idx"]
            - bounds["start_idx"]
        )

        export_exceeds_recording = (
            bounds["export_end_idx"]
            > len(df)
        )

        b0_results.append(
            {
                "participant": row["participant"],
                "trial_number": row["trial_number"],
                "rep_number": row["rep_number"],
                "file": row["file"],
                "start_idx": bounds["start_idx"],
                "end_idx": bounds["end_idx"],
                "export_end_idx": bounds["export_end_idx"],
                "movement_length": movement_length,
                "export_length": export_length,
                "export_exceeds_recording": export_exceeds_recording,
            }
        )

    except Exception as exc:

        b0_errors.append(
            {
                "file": row["file"],
                "error": str(exc),
            }
        )


b0_results_df = pd.DataFrame(
    b0_results
)

b0_errors_df = pd.DataFrame(
    b0_errors
)


# ---------------------------------------------------------
# Basic extraction checks
# ---------------------------------------------------------

print(
    "Retained B0 recordings:",
    len(b0_df),
)

print(
    "Start/end pairs computed:",
    len(b0_results_df),
)

print(
    "Extraction exceptions:",
    len(b0_errors_df),
)

print(
    "Export intervals exceeding recording length:",
    b0_results_df[
        "export_exceeds_recording"
    ].sum(),
)


# ---------------------------------------------------------
# Ordering check
# ---------------------------------------------------------

ordered_bounds = (
    b0_results_df["start_idx"]
    < b0_results_df["end_idx"]
)

print(
    "Ordered start/end pairs:",
    ordered_bounds.sum(),
    "/",
    len(b0_results_df),
)


# ---------------------------------------------------------
# Movement-length summary
# ---------------------------------------------------------

print(
    "\nDetected movement-length summary:"
)

display(
    b0_results_df[
        "movement_length"
    ]
    .describe()
    .to_frame(
        name="samples"
    )
)


# ---------------------------------------------------------
# Export-length summary
# ---------------------------------------------------------

print(
    "\nExported trajectory-length summary:"
)

display(
    b0_results_df[
        "export_length"
    ]
    .describe()
    .to_frame(
        name="samples"
    )
)


# ---------------------------------------------------------
# Compact key statistics
# ---------------------------------------------------------

movement_q1 = b0_results_df[
    "movement_length"
].quantile(0.25)

movement_median = b0_results_df[
    "movement_length"
].median()

movement_q3 = b0_results_df[
    "movement_length"
].quantile(0.75)

movement_min = b0_results_df[
    "movement_length"
].min()

movement_max = b0_results_df[
    "movement_length"
].max()

export_median = b0_results_df[
    "export_length"
].median()

export_min = b0_results_df[
    "export_length"
].min()

export_max = b0_results_df[
    "export_length"
].max()


print(
    "\nCompact summary:"
)

print(
    f"Detected movement median: {movement_median:.0f} samples"
)

print(
    f"Detected movement middle 50%: "
    f"{movement_q1:.0f}-{movement_q3:.0f} samples"
)

print(
    f"Detected movement range: "
    f"{movement_min:.0f}-{movement_max:.0f} samples"
)

print(
    f"Exported trajectory median: "
    f"{export_median:.0f} samples"
)

print(
    f"Exported trajectory range: "
    f"{export_min:.0f}-{export_max:.0f} samples"
)


if not b0_errors_df.empty:

    print(
        "\nExtraction errors:"
    )

    display(
        b0_errors_df
    )

/Users/arthu/miniconda3/envs/ctm_robotics/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:1317: RuntimeWarning: divide by zero encountered in divide
  a = -(dx2) / (dx1 * (dx1 + dx2))
/Users/arthu/miniconda3/envs/ctm_robotics/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:1318: RuntimeWarning: divide by zero encountered in divide
  b = (dx2 - dx1) / (dx1 * dx2)
/Users/arthu/miniconda3/envs/ctm_robotics/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:1319: RuntimeWarning: divide by zero encountered in divide
  c = dx1 / (dx2 * (dx1 + dx2))
/Users/arthu/miniconda3/envs/ctm_robotics/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:1328: RuntimeWarning: invalid value encountered in multiply
  out[tuple(slice1)] = a * f[tuple(slice2)] + b * f[tuple(slice3)] \
/Users/arthu/miniconda3/envs/ctm_robotics/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:1328: RuntimeWarning: invalid value encountered in add
  out[tuple(slice1)]

Retained B0 recordings: 318
Start/end pairs computed: 318
Extraction exceptions: 0
Export intervals exceeding recording length: 0
Ordered start/end pairs: 318 / 318

Detected movement-length summary:


,samples
count,318.000000
mean,83.028302
std,20.516112
min,8.000000
25%,72.000000
50%,81.000000
75%,91.000000
max,247.000000



Exported trajectory-length summary:


,samples
count,318.000000
mean,133.028302
std,20.516112
min,58.000000
25%,122.000000
50%,131.000000
75%,141.000000
max,297.000000



Compact summary:
Detected movement median: 81 samples
Detected movement middle 50%: 72-91 samples
Detected movement range: 8-247 samples
Exported trajectory median: 131 samples
Exported trajectory range: 58-297 samples


## 5. Flagged Recordings and Methodological Limits

The B0 extraction is computationally reproducible across all 318 retained recordings, but this does not imply that every computed interval is physically meaningful.

Several recordings were therefore flagged for closer inspection rather than automatically excluded.

### 5.1 Flagged B0 recordings

The main boundary outliers identified so far are:

| Participant | Trial | Rep | Detected length | Main observation |
|---:|---:|---:|---:|---|
| 5 | 4 | 0 | 8 samples | Very small board threshold causes an early endpoint |
| 3 | 5 | 2 | 20 samples | Glove speed is already high at the beginning of the recording |
| 4 | 6 | 6 | 247 samples | Glove-speed onset occurs much earlier than `hand_active` |
| 2 | 1 | 7 | 217 samples | Clear early glove-speed increase relative to object lift |

These recordings are retained because their unusual boundaries can be explained by the original preprocessing rule.

No additional exclusion criterion has been introduced.

The current principle is:

    reproduce the original method
              |
              v
        identify unusual cases
              |
              v
      evaluate them explicitly
              |
              v
    modify preprocessing only if justified

### 5.2 Participant 5 short-interval case

The shortest detected movement occurs in:

    participant 5
    B0_trial4_rep0_attempt0.csv

with:

    start_idx = 1
    end_idx = 9
    movement_length = 8

For this recording, the B0 endpoint threshold is only:

$$
\theta_{\mathrm{end}} = 1.
$$

The original method computes the threshold from:

$$
0.2 \max b(t),
$$

while the threshold-crossing helper tests:

$$
|b(t)| > \theta_{\mathrm{end}}.
$$

This means that a small negative board fluctuation can trigger the endpoint when the positive maximum is very small.

The recording also has:

    hand_active index = 0

so the recording itself already begins in an unusual state.

This case is computationally reproducible, but its physical interpretation as a complete reach-to-grasp interval remains questionable.

### 5.3 Early glove-speed onset

For:

    participant 3
    B0_trial5_rep2_attempt0.csv

the detected interval is only 20 samples long.

Here, the board endpoint corresponds to a substantial board-signal increase, but the glove-speed profile is already large at the beginning of the recording.

The resulting start is:

    start_idx = 1

while:

    hand_active index = 20

This illustrates that the glove-speed criterion and `hand_active` represent different onset definitions.

The same distinction is visible in the long participant-4 recording:

    participant 4
    B0_trial6_rep6_attempt0.csv

where:

    start_idx = 62
    hand_active index = 257

The speed profile shows sustained movement around the detected start, so the early start does not appear to be caused by a single isolated numerical spike.

### 5.4 Duplicate glove timestamp

Exactly one retained B0 recording contains an adjacent duplicate glove timestamp:

    participant 5
    B0_trial2_rep0_attempt0.csv

This produces NumPy warnings during:

    np.gradient(data, timestamps, axis=0)

because a repeated timestamp implies:

$$
\Delta t = 0.
$$

However, the duplicate occurs after the detected B0 endpoint.

Therefore, it is outside the region used to calculate:

    speed[:end]

and does not affect the B0 start threshold for this recording.

The timestamp sequence is currently left unchanged so that the reproduced preprocessing remains faithful to the original implementation.

### 5.5 Exported trajectories are not identical to detected reaches

The original export retains 50 samples after the detected endpoint.

Therefore:

    detected movement interval

and:

    exported trajectory

must remain conceptually separate.

For the median B0 example:

    detected interval = 81 samples
    exported interval = 131 samples

Approximately 38% of the exported sample count therefore lies in the added 50-sample portion.

This becomes especially important for later CTM window experiments.

For example, the final 25% of a normalized exported trajectory should not automatically be described as:

    the final 25% of the reach

because part of that interval may occur after the detected board-based endpoint.

Window positions should therefore be described relative to the exported trajectory unless a separate physical-reach normalization is introduced.

### 5.6 Current resampling convention

The current `resample_trajectory()` function performs linear interpolation over normalized sample position.

Conceptually, an input trajectory with $N$ rows is assigned coordinates spanning:

$$
0 \rightarrow 1
$$

according to row position, and the trajectory is interpolated onto a new fixed grid.

This means that the current operation is best described as:

    linear interpolation over normalized sample index

rather than:

    interpolation over physical timestamps

The method implicitly assumes that adjacent input samples are approximately equally spaced.

This is a reasonable explicit preprocessing choice for constructing fixed-size CTM inputs, but it should not be confused with physical-time interpolation.

Before making claims about timing in milliseconds or fractions of physical movement duration, interpolation using the recorded glove timestamps should be considered separately.

### 5.7 Current interpretation

The current loader therefore distinguishes three different questions:

    Can the original preprocessing rule be reproduced?

    Does the rule produce ordered numerical boundaries?

    Are those boundaries physically meaningful for every recording?

For B0, the answers currently are:

    reproduction: yes
    ordered boundaries: 318 / 318
    physical plausibility: generally plausible, with flagged exceptions

The flagged recordings remain part of the dataset for now and should remain visible during later CTM validation rather than being silently removed.

In [7]:
from pathlib import Path
import numpy as np
import pandas as pd

from data_loader import (
    DATA_ROOT,
    FINGERTIP_COLUMNS,
    resample_trajectory,
)


# ---------------------------------------------------------
# 1. Compact table of the four flagged B0 recordings
# ---------------------------------------------------------

flagged_cases = [
    (5, 4, 0),
    (3, 5, 2),
    (4, 6, 6),
    (2, 1, 7),
]

flagged_mask = pd.Series(
    False,
    index=b0_results_df.index,
)

for participant, trial, rep in flagged_cases:

    flagged_mask |= (
        (b0_results_df["participant"] == participant)
        & (b0_results_df["trial_number"] == trial)
        & (b0_results_df["rep_number"] == rep)
    )


flagged_b0_df = (
    b0_results_df[
        flagged_mask
    ]
    [
        [
            "participant",
            "trial_number",
            "rep_number",
            "file",
            "start_idx",
            "end_idx",
            "export_end_idx",
            "movement_length",
            "export_length",
        ]
    ]
    .sort_values(
        [
            "movement_length",
            "participant",
        ]
    )
)


print("Flagged B0 recordings:")
display(flagged_b0_df)


# ---------------------------------------------------------
# 2. Audit duplicate glove timestamps
# ---------------------------------------------------------

duplicate_records = []

for _, row in b0_results_df.iterrows():

    path = (
        DATA_ROOT
        / f"participant_{int(row['participant'])}"
        / "motion"
        / row["file"]
    )

    df = pd.read_csv(path)

    glove_time = df[
        "glove_timestamps"
    ].to_numpy(
        dtype=float
    )

    duplicate_indices = np.where(
        np.diff(glove_time) == 0
    )[0]

    if len(duplicate_indices) > 0:

        duplicate_before_end = np.sum(
            duplicate_indices
            < row["end_idx"] - 1
        )

        duplicate_records.append(
            {
                "participant": row["participant"],
                "trial_number": row["trial_number"],
                "rep_number": row["rep_number"],
                "file": row["file"],
                "n_duplicate_steps": len(
                    duplicate_indices
                ),
                "duplicate_indices": (
                    duplicate_indices.tolist()
                ),
                "duplicates_before_end": int(
                    duplicate_before_end
                ),
            }
        )


duplicate_df = pd.DataFrame(
    duplicate_records
)


print(
    "\nB0 recordings with duplicate glove timestamps:",
    len(duplicate_df),
)

if not duplicate_df.empty:
    display(duplicate_df)


# ---------------------------------------------------------
# 3. Crop the 15-channel exported fingertip trajectories
#    and resample each one to the fixed CTM length
# ---------------------------------------------------------

resampling_checks = []

for _, row in b0_results_df.iterrows():

    path = (
        DATA_ROOT
        / f"participant_{int(row['participant'])}"
        / "motion"
        / row["file"]
    )

    df = pd.read_csv(path)

    fingertip_data = df[
        FINGERTIP_COLUMNS
    ].to_numpy(
        dtype=np.float32
    )

    start = int(
        row["start_idx"]
    )

    export_end = int(
        row["export_end_idx"]
    )

    exported = fingertip_data[
        start:export_end
    ]

    resampled = resample_trajectory(
        exported
    )

    resampling_checks.append(
        {
            "participant": row["participant"],
            "trial_number": row["trial_number"],
            "rep_number": row["rep_number"],
            "file": row["file"],
            "input_length": len(exported),
            "input_finite": bool(
                np.isfinite(
                    exported
                ).all()
            ),
            "resampled_shape": (
                tuple(
                    resampled.shape
                )
            ),
            "resampled_finite": bool(
                np.isfinite(
                    resampled
                ).all()
            ),
        }
    )


resampling_df = pd.DataFrame(
    resampling_checks
)


# ---------------------------------------------------------
# 4. Validate all exported/resampled B0 trajectories
# ---------------------------------------------------------

print(
    "\nExported fingertip trajectories checked:",
    len(resampling_df),
)

print(
    "Finite exported trajectories:",
    resampling_df[
        "input_finite"
    ].sum(),
    "/",
    len(resampling_df),
)

print(
    "Finite resampled trajectories:",
    resampling_df[
        "resampled_finite"
    ].sum(),
    "/",
    len(resampling_df),
)


print(
    "\nResampled shapes:"
)

display(
    resampling_df[
        "resampled_shape"
    ]
    .value_counts()
    .rename(
        "n_recordings"
    )
    .to_frame()
)


print(
    "\nExported input-length summary:"
)

display(
    resampling_df[
        "input_length"
    ]
    .describe()
    .to_frame(
        name="samples"
    )
)


# ---------------------------------------------------------
# 5. Show failures if any exist
# ---------------------------------------------------------

invalid_resampled = resampling_df[
    ~resampling_df[
        "resampled_finite"
    ]
]

if not invalid_resampled.empty:

    print(
        "\nNon-finite resampled trajectories:"
    )

    display(
        invalid_resampled
    )

Flagged B0 recordings:


,participant,trial_number,rep_number,file,start_idx,end_idx,export_end_idx,movement_length,export_length
286,5,4,0,B0_trial4_rep0_attempt0.csv,1,9,59,8,58
168,3,5,2,B0_trial5_rep2_attempt0.csv,1,21,71,20,70
79,2,1,7,B0_trial1_rep7_attempt0.csv,92,309,359,217,267
244,4,6,6,B0_trial6_rep6_attempt0.csv,62,309,359,247,297



B0 recordings with duplicate glove timestamps: 1


,participant,trial_number,rep_number,file,n_duplicate_steps,duplicate_indices,duplicates_before_end
0,5,2,0,B0_trial2_rep0_attempt0.csv,1,[432],0



Exported fingertip trajectories checked: 318
Finite exported trajectories: 318 / 318
Finite resampled trajectories: 318 / 318

Resampled shapes:


,n_recordings
resampled_shape,
"(15, 100)",318



Exported input-length summary:


,samples
count,318.000000
mean,133.028302
std,20.516112
min,58.000000
25%,122.000000
50%,131.000000
75%,141.000000
max,297.000000


In [8]:
# ---------------------------------------------------------
# Metadata consistency audit
# ---------------------------------------------------------
#
# The inventory obtains block/trial/rep/attempt identifiers
# from the filename.
#
# extract_raw_metadata() obtains the same identifiers from
# the CSV contents.
#
# Before moving to B1-B4, I will verify that both representations
# agree for every physical recording.
# ---------------------------------------------------------

identifier_columns = [
    "block",
    "trial_number",
    "rep_number",
    "attempt_number",
]

consistency_records = []


for _, inventory_row in inventory.iterrows():

    path = Path(
        inventory_row["path"]
    )

    df = pd.read_csv(
        path,
        usecols=identifier_columns,
    )

    record = {
        "participant": inventory_row["participant"],
        "file": inventory_row["file"],
    }

    recording_consistent = True

    for column in identifier_columns:

        # Identifier parsed from the filename
        filename_value = int(
            inventory_row[column]
        )

        # All valid values appearing inside the CSV
        csv_values = (
            df[column]
            .dropna()
            .unique()
        )

        # The identifier should remain constant throughout
        # one recording.
        is_constant = (
            len(csv_values) == 1
        )

        if is_constant:
            csv_value = int(
                csv_values[0]
            )

            matches_filename = (
                csv_value
                == filename_value
            )

        else:
            csv_value = None
            matches_filename = False

        record[
            f"{column}_filename"
        ] = filename_value

        record[
            f"{column}_csv"
        ] = csv_value

        record[
            f"{column}_constant"
        ] = is_constant

        record[
            f"{column}_matches"
        ] = matches_filename

        if (
            not is_constant
            or not matches_filename
        ):
            recording_consistent = False

    record[
        "all_identifiers_consistent"
    ] = recording_consistent

    consistency_records.append(
        record
    )


metadata_consistency_df = pd.DataFrame(
    consistency_records
)


# ---------------------------------------------------------
# Summary
# ---------------------------------------------------------

print(
    "Physical recordings checked:",
    len(metadata_consistency_df),
)

print(
    "Fully consistent recordings:",
    metadata_consistency_df[
        "all_identifiers_consistent"
    ].sum(),
    "/",
    len(metadata_consistency_df),
)


for column in identifier_columns:

    print(
        f"\n{column}:"
    )

    print(
        "  constant within recording:",
        metadata_consistency_df[
            f"{column}_constant"
        ].sum(),
        "/",
        len(metadata_consistency_df),
    )

    print(
        "  matches filename:",
        metadata_consistency_df[
            f"{column}_matches"
        ].sum(),
        "/",
        len(metadata_consistency_df),
    )


# ---------------------------------------------------------
# Show any mismatches
# ---------------------------------------------------------

inconsistent_df = metadata_consistency_df[
    ~metadata_consistency_df[
        "all_identifiers_consistent"
    ]
]

print(
    "\nRecordings with identifier inconsistencies:",
    len(inconsistent_df),
)

if not inconsistent_df.empty:

    display(
        inconsistent_df
    )

Physical recordings checked: 976
Fully consistent recordings: 976 / 976

block:
  constant within recording: 976 / 976
  matches filename: 976 / 976

trial_number:
  constant within recording: 976 / 976
  matches filename: 976 / 976

rep_number:
  constant within recording: 976 / 976
  matches filename: 976 / 976

attempt_number:
  constant within recording: 976 / 976
  matches filename: 976 / 976

Recordings with identifier inconsistencies: 0


## 6. Current Status and B1-B4 Next Steps

The current preprocessing pipeline has reached a stable intermediate milestone.

The metadata handling and B0 reconstruction are now both reproducible from the notebook.

### 6.1 Completed components

The following parts are currently implemented and validated:

- 976 physical processed recording files were inventoried.
- These correspond to 960 logical recording slots.
- Sixteen retry slots contain a failed attempt followed by a successful retry.
- Two B0 slots have no successful replacement.
- The retained dataset therefore contains 958 successful recordings.
- The successful-attempt selection logic has been validated across the complete dataset.
- `target_object` changes within some recordings are handled using the final valid non-missing value.
- The B1-B4 repetition-0 correction is implemented as a repetition-1 lookup.
- The correction is applied to 160 retained recordings.
- None of the 160 corrected values changes numerically in the released processed dataset.
- B0 class labels are derived from `trial_number`.
- B0 uses the fixed board channel `board_data_3`.
- B1-B4 class labels are derived from corrected `target_object`.
- B1-B4 board channels are selected using `trial_number`.
- The resulting class distribution matches the expected experimental structure.
- The original threshold-crossing helper has been reproduced.
- The original glove-speed calculation has been reproduced.
- The original board-saturation cleanup has been reproduced.
- B0 movement-boundary extraction has been reconstructed from the original source code.
- All 318 retained B0 recordings produce ordered start/end indices without extraction exceptions.
- No B0 `export_end_idx` exceeds the available recording length.
- All 318 exported 15-channel fingertip trajectories contain finite values.
- All 318 currently resampled B0 trajectories contain finite values.
- The filename-derived and CSV-derived recording identifiers agree for all 976 physical files.
- `block`, `trial_number`, `rep_number`, and `attempt_number` are constant within every recording.

The metadata consistency audit therefore produced:

    physical recordings checked = 976
    fully consistent recordings = 976
    identifier inconsistencies = 0

This provides an additional safeguard against accidental misgrouping before the loader is expanded further.

### 6.2 B0 status

The B0 extraction currently reproduces the behaviour of the original movement-export code.

Across the 318 retained B0 recordings:

    median detected interval = 81 samples
    middle 50% = 72-91 samples
    full detected range = 8-247 samples

The original export adds 50 samples after the detected endpoint:

    median exported interval = 131 samples
    full exported range = 58-297 samples

Several unusual recordings remain explicitly flagged.

These include very short computed intervals, unusually long intervals, and one duplicate glove timestamp.

The current interpretation is:

    original B0 extraction is reproducible

but not:

    every computed boundary is necessarily physically optimal

No additional exclusion rule has been introduced at this stage.

### 6.3 Current resampling convention

The current trajectory resampling function performs linear interpolation over normalized sample index.

It does not currently interpolate using the recorded glove timestamps.

Therefore, fixed-length CTM trajectories produced by the current helper should be described as:

    sample-index normalized trajectories

rather than:

    physical-time normalized trajectories

This distinction should remain explicit when interpreting later window-position or partial-observation experiments.

### 6.4 Next preprocessing task

The major unresolved component is the movement-extraction path for B1-B4.

The next step is therefore to inspect the original B1-B4 evaluation and preprocessing code and determine exactly how the relevant movement interval should be defined.

The immediate workflow is:

    inspect B1-B4 source-code path
              |
              v
    identify start and endpoint definitions
              |
              v
    reproduce them in a separate loader function
              |
              v
    validate all 640 retained B1-B4 recordings
              |
              v
    compare B0 and B1-B4 extraction semantics
              |
              v
    define the final unified CTM sample builder

The B1-B4 implementation should be validated before introducing a single high-level function that constructs the complete real-data CTM dataset.

### 6.5 Planned final loader output

Once B1-B4 preprocessing is validated, each retained recording should ultimately provide at least:

    participant
    block
    trial_number
    rep_number
    class_label
    board_column
    start_idx
    end_idx
    export_end_idx
    trajectory
    preprocessing diagnostics

The final trajectory representation is expected to contain:

$$
15
$$

fingertip channels and a fixed number of resampled steps.

The exact resampling convention should remain explicit in the dataset metadata.

### 6.6 Experimental work after loader completion

After the full real-data loader is validated, the first CTM experiment should be a small pilot rather than the full synthetic-style experiment grid.

The initial real-data pilot should verify that:

- the loader produces the expected tensor shapes,
- train/validation/test splitting works correctly,
- class labels remain balanced enough for evaluation,
- the model trains without data-shape or numerical problems,
- and the resulting behaviour is sensible before more complex perturbation experiments are introduced.

Only after this pilot should the larger experiments be extended to the reach-to-grasp dataset, including:

    partial observation
    window position
    correlated noise
    internal CTM ticks
    certainty behaviour

This notebook therefore currently represents a completed milestone for:

    dataset reconstruction
    metadata validation
    B0 preprocessing reproduction

while:

    B1-B4 movement extraction
    final unified loader
    CTM real-data pilot

remain the next stages.

In [9]:
from pathlib import Path

import numpy as np
import pandas as pd

from data_loader import (
    first_change_index,
    replace_with_highest,
)


# ---------------------------------------------------------
# B1-B4 boundary-definition audit
# ---------------------------------------------------------
#
# This cell does NOT define the final CTM crop.
#
# It compares three candidate movement-onset definitions:
#
#     1. first hand_active == 1
#     2. palm sensor crossing 10% of its maximum displacement
#     3. palm sensor displacement crossing 50 mm
#
# The endpoint reproduces the reach-to-grasp construction
# used in Fig17.py:
#
#     start from first hand_active
#     select board_data_{trial_number}
#     zero-baseline the board signal at that start
#     detect the first 10%-of-maximum crossing
#
# No source CSV is modified.
# ---------------------------------------------------------


BOARD_COLUMNS = [
    f"board_data_{i}"
    for i in range(8)
]


def boundary_candidates_b1_b4(df, board_sensor_index):
    """
    Compute candidate B1-B4 movement boundaries.

    This is a diagnostic helper only.

    Returns
    -------
    dict
        Candidate onset indices and the Fig17-style
        object-lift endpoint.
    """

    # -----------------------------------------------------
    # 1. hand_active onset
    # -----------------------------------------------------

    hand_active = df[
        "hand_active"
    ].to_numpy()

    hand_active_indices = np.where(
        hand_active == 1
    )[0]

    if len(hand_active_indices) == 0:
        start_hand_active = None
    else:
        start_hand_active = int(
            hand_active_indices[0]
        )


    # -----------------------------------------------------
    # 2. Palm signal
    # -----------------------------------------------------
    #
    # Reproduce the cleanup used in the evaluation code:
    #
    #     replace 8191
    #     replace 8190
    #     subtract first value
    #
    # first_change_index() itself uses abs(signal).
    # -----------------------------------------------------

    palm_data = df[
        "palm_data"
    ].to_numpy(
        dtype=float
    )

    palm_data = replace_with_highest(
        palm_data,
        target=8191.0,
    )

    palm_data = replace_with_highest(
        palm_data,
        target=8190.0,
    )

    palm_relative = (
        palm_data
        - palm_data[0]
    )


    # Original evaluation-style palm event:
    # 10% of maximum relative palm displacement

    palm_10_threshold = (
        0.1
        * palm_relative.max()
    )

    start_palm_10pct = first_change_index(
        palm_relative,
        palm_10_threshold,
    )


    # Manuscript-motivated diagnostic:
    # 50 mm displacement from rest

    start_palm_50mm = first_change_index(
        palm_relative,
        50.0,
    )


    # -----------------------------------------------------
    # 3. Fig17-style object-lift endpoint
    # -----------------------------------------------------
    #
    # Fig17.py starts from hand_active and then:
    #
    #     board = board_data[start:-1, trial_number]
    #     board -= board[0]
    #     t_lift = first_change_index(
    #         board,
    #         0.1 * board.max()
    #     )
    #
    # t_lift is relative to start, so convert it back
    # to an absolute recording index.
    # -----------------------------------------------------

    board_data = df[
        BOARD_COLUMNS
    ].to_numpy(
        dtype=float
    )

    board_data = replace_with_highest(
        board_data,
        target=8191.0,
    )

    board_data = replace_with_highest(
        board_data,
        target=8190.0,
    )


    end_board_10pct = None
    board_10_threshold = None
    board_max_displacement = None
    t_lift_relative = None


    if start_hand_active is not None:

        board_signal = board_data[
            start_hand_active:-1,
            int(board_sensor_index),
        ]

        if len(board_signal) > 1:

            board_relative = (
                board_signal
                - board_signal[0]
            )

            board_max_displacement = float(
                board_relative.max()
            )

            board_10_threshold = float(
                0.1
                * board_max_displacement
            )

            t_lift_relative = first_change_index(
                board_relative,
                board_10_threshold,
            )

            if t_lift_relative is not None:

                end_board_10pct = int(
                    start_hand_active
                    + t_lift_relative
                )


    return {
        "start_hand_active": start_hand_active,
        "start_palm_10pct": start_palm_10pct,
        "start_palm_50mm": start_palm_50mm,
        "end_board_10pct": end_board_10pct,
        "palm_10_threshold": float(
            palm_10_threshold
        ),
        "board_10_threshold": board_10_threshold,
        "board_max_displacement": board_max_displacement,
        "t_lift_relative_to_hand_active": t_lift_relative,
    }


# ---------------------------------------------------------
# Run audit over all retained B1-B4 recordings
# ---------------------------------------------------------

b1_b4_df = mapped_df[
    mapped_df["block"].isin(
        [1, 2, 3, 4]
    )
].copy()


boundary_records = []
boundary_errors = []


for _, row in b1_b4_df.iterrows():

    path = Path(
        row["path"]
    )

    try:

        df = pd.read_csv(
            path
        )

        bounds = boundary_candidates_b1_b4(
            df=df,
            board_sensor_index=row[
                "board_sensor_index"
            ],
        )

        glove_time = df[
            "glove_timestamps"
        ].to_numpy(
            dtype=float
        )


        record = {
            "participant": int(
                row["participant"]
            ),
            "block": int(
                row["block"]
            ),
            "trial_number": int(
                row["trial_number"]
            ),
            "rep_number": int(
                row["rep_number"]
            ),
            "file": row["file"],
            "class_label": int(
                row["class_label"]
            ),
            "board_column": row[
                "board_column"
            ],
            **bounds,
        }


        # -------------------------------------------------
        # Candidate ordering
        # -------------------------------------------------

        end = bounds[
            "end_board_10pct"
        ]

        for start_name in [
            "start_hand_active",
            "start_palm_10pct",
            "start_palm_50mm",
        ]:

            start = bounds[
                start_name
            ]

            prefix = start_name.replace(
                "start_",
                "",
            )

            if (
                start is not None
                and end is not None
            ):

                record[
                    f"{prefix}_ordered"
                ] = bool(
                    start < end
                )

                record[
                    f"{prefix}_movement_length"
                ] = int(
                    end - start
                )

            else:

                record[
                    f"{prefix}_ordered"
                ] = False

                record[
                    f"{prefix}_movement_length"
                ] = np.nan


        # -------------------------------------------------
        # Difference between onset definitions
        #
        # Positive:
        #     first candidate occurs later than second.
        #
        # Negative:
        #     first candidate occurs earlier.
        # -------------------------------------------------

        ha = bounds[
            "start_hand_active"
        ]

        p10 = bounds[
            "start_palm_10pct"
        ]

        p50 = bounds[
            "start_palm_50mm"
        ]


        def index_difference(a, b):
            if a is None or b is None:
                return np.nan

            return int(
                a - b
            )


        def time_difference_ms(a, b):
            if a is None or b is None:
                return np.nan

            if (
                a >= len(glove_time)
                or b >= len(glove_time)
            ):
                return np.nan

            return float(
                glove_time[a]
                - glove_time[b]
            )


        record[
            "palm10_minus_hand_active_samples"
        ] = index_difference(
            p10,
            ha,
        )

        record[
            "palm50_minus_hand_active_samples"
        ] = index_difference(
            p50,
            ha,
        )

        record[
            "palm50_minus_palm10_samples"
        ] = index_difference(
            p50,
            p10,
        )


        record[
            "palm10_minus_hand_active_ms"
        ] = time_difference_ms(
            p10,
            ha,
        )

        record[
            "palm50_minus_hand_active_ms"
        ] = time_difference_ms(
            p50,
            ha,
        )

        record[
            "palm50_minus_palm10_ms"
        ] = time_difference_ms(
            p50,
            p10,
        )


        boundary_records.append(
            record
        )


    except Exception as exc:

        boundary_errors.append(
            {
                "participant": row[
                    "participant"
                ],
                "block": row[
                    "block"
                ],
                "trial_number": row[
                    "trial_number"
                ],
                "rep_number": row[
                    "rep_number"
                ],
                "file": row[
                    "file"
                ],
                "error": str(
                    exc
                ),
            }
        )


b1_b4_boundaries_df = pd.DataFrame(
    boundary_records
)

b1_b4_boundary_errors_df = pd.DataFrame(
    boundary_errors
)


# ---------------------------------------------------------
# Detection success
# ---------------------------------------------------------

print(
    "Retained B1-B4 recordings:",
    len(b1_b4_df),
)

print(
    "Boundary records computed:",
    len(b1_b4_boundaries_df),
)

print(
    "Exceptions:",
    len(b1_b4_boundary_errors_df),
)


for column in [
    "start_hand_active",
    "start_palm_10pct",
    "start_palm_50mm",
    "end_board_10pct",
]:

    print(
        f"{column}:",
        b1_b4_boundaries_df[
            column
        ].notna().sum(),
        "/",
        len(
            b1_b4_boundaries_df
        ),
    )


# ---------------------------------------------------------
# Ordering relative to object lift
# ---------------------------------------------------------

print(
    "\nOrdered onset -> object-lift pairs:"
)

for column in [
    "hand_active_ordered",
    "palm_10pct_ordered",
    "palm_50mm_ordered",
]:

    print(
        f"{column}:",
        b1_b4_boundaries_df[
            column
        ].sum(),
        "/",
        len(
            b1_b4_boundaries_df
        ),
    )


# ---------------------------------------------------------
# Timing differences
# ---------------------------------------------------------

print(
    "\nOnset-definition differences in milliseconds:"
)

display(
    b1_b4_boundaries_df[
        [
            "palm10_minus_hand_active_ms",
            "palm50_minus_hand_active_ms",
            "palm50_minus_palm10_ms",
        ]
    ]
    .describe()
    .T
)


# ---------------------------------------------------------
# Candidate movement-length distributions
# ---------------------------------------------------------

print(
    "\nCandidate movement lengths in samples:"
)

display(
    b1_b4_boundaries_df[
        [
            "hand_active_movement_length",
            "palm_10pct_movement_length",
            "palm_50mm_movement_length",
        ]
    ]
    .describe()
    .T
)


# ---------------------------------------------------------
# Largest hand_active vs 50-mm disagreements
# ---------------------------------------------------------

largest_disagreements = (
    b1_b4_boundaries_df
    .assign(
        absolute_difference_ms=lambda x:
        x[
            "palm50_minus_hand_active_ms"
        ].abs()
    )
    .sort_values(
        "absolute_difference_ms",
        ascending=False,
    )
    [
        [
            "participant",
            "block",
            "trial_number",
            "rep_number",
            "file",
            "start_hand_active",
            "start_palm_10pct",
            "start_palm_50mm",
            "end_board_10pct",
            "palm10_minus_hand_active_ms",
            "palm50_minus_hand_active_ms",
            "absolute_difference_ms",
        ]
    ]
    .head(15)
)


print(
    "\nLargest hand_active / palm-50-mm disagreements:"
)

display(
    largest_disagreements
)


if not b1_b4_boundary_errors_df.empty:

    print(
        "\nBoundary extraction errors:"
    )

    display(
        b1_b4_boundary_errors_df
    )

Retained B1-B4 recordings: 640
Boundary records computed: 640
Exceptions: 0
start_hand_active: 640 / 640
start_palm_10pct: 640 / 640
start_palm_50mm: 640 / 640
end_board_10pct: 640 / 640

Ordered onset -> object-lift pairs:
hand_active_ordered: 640 / 640
palm_10pct_ordered: 640 / 640
palm_50mm_ordered: 640 / 640

Onset-definition differences in milliseconds:


,count,mean,std,min,25%,50%,75%,max
palm10_minus_hand_active_ms,640.0,-240.453125,200.974054,-3343.0,-266.0,-217.5,-168.25,0.0
palm50_minus_hand_active_ms,640.0,-128.495313,137.669025,-1431.0,-142.0,-110.0,-64.00,0.0
palm50_minus_palm10_ms,640.0,111.957813,151.973856,-95.0,64.0,94.0,126.00,3327.0



Candidate movement lengths in samples:


,count,mean,std,min,25%,50%,75%,max
hand_active_movement_length,640.0,60.937500,20.473025,2.0,51.0,60.0,71.0,257.0
palm_10pct_movement_length,640.0,76.212500,20.199285,24.0,64.0,74.0,85.0,266.0
palm_50mm_movement_length,640.0,69.103125,19.309002,9.0,59.0,68.0,78.0,261.0



Largest hand_active / palm-50-mm disagreements:


,participant,block,trial_number,rep_number,file,start_hand_active,start_palm_10pct,start_palm_50mm,end_board_10pct,palm10_minus_hand_active_ms,palm50_minus_hand_active_ms,absolute_difference_ms
175,2,2,3,3,B2_trial3_rep3_attempt0.csv,361,260,270,363,-1588.0,-1431.0,1431.0
174,2,2,3,2,B2_trial3_rep2_attempt0.csv,417,320,328,420,-1534.0,-1406.0,1406.0
173,2,2,3,1,B2_trial3_rep1_attempt0.csv,321,228,232,324,-1471.0,-1406.0,1406.0
300,3,2,3,0,B2_trial3_rep0_attempt0.csv,294,204,210,296,-1415.0,-1321.0,1321.0
172,2,2,3,0,B2_trial3_rep0_attempt0.csv,177,92,100,180,-1334.0,-1210.0,1210.0
237,2,4,3,1,B4_trial3_rep1_attempt0.csv,262,202,206,266,-934.0,-872.0,872.0
236,2,4,3,0,B4_trial3_rep0_attempt0.csv,277,226,233,300,-806.0,-695.0,695.0
142,2,1,3,2,B1_trial3_rep2_attempt1.csv,270,230,234,298,-628.0,-566.0,566.0
527,5,1,3,3,B1_trial3_rep3_attempt0.csv,252,208,218,291,-686.0,-530.0,530.0
591,5,3,3,3,B3_trial3_rep3_attempt0.csv,245,209,213,286,-567.0,-503.0,503.0


In [13]:
# ---------------------------------------------------------
# B1-B4 endpoint sensitivity to onset definition
# ---------------------------------------------------------
#
# For each retained B1-B4 recording, detect the board 10%
# lift endpoint independently after:
#
#   1. recording start = 0
#   2. first hand_active == 1
#   3. palm 10% onset
#   4. palm 50 mm onset
#
# No loader modification yet.
# ---------------------------------------------------------

from pathlib import Path

import numpy as np
import pandas as pd


# ---------------------------------------------------------
# Restore operational metadata
# ---------------------------------------------------------
#
# b1_b4_boundaries_df currently contains compact filenames,
# but we need the full source path to read the CSV again.
# We also restore board_sensor_index directly instead of
# reconstructing it from the board column string.
# ---------------------------------------------------------

path_lookup = (
    b1_b4_df[
        [
            "participant",
            "block",
            "trial_number",
            "rep_number",
            "file",
            "path",
            "board_sensor_index",
        ]
    ]
    .drop_duplicates()
)


# Avoid duplicate columns if this cell is rerun.
for col in [
    "path",
    "board_sensor_index",
]:
    if col in b1_b4_boundaries_df.columns:
        b1_b4_boundaries_df = (
            b1_b4_boundaries_df
            .drop(columns=[col])
        )


b1_b4_boundaries_df = (
    b1_b4_boundaries_df
    .merge(
        path_lookup,
        on=[
            "participant",
            "block",
            "trial_number",
            "rep_number",
            "file",
        ],
        how="left",
        validate="one_to_one",
    )
)


print(
    "Resolved paths:",
    b1_b4_boundaries_df[
        "path"
    ].notna().sum(),
    "/",
    len(
        b1_b4_boundaries_df
    ),
)

print(
    "Resolved board sensor indices:",
    b1_b4_boundaries_df[
        "board_sensor_index"
    ].notna().sum(),
    "/",
    len(
        b1_b4_boundaries_df
    ),
)


assert (
    b1_b4_boundaries_df[
        "path"
    ].notna().all()
)

assert (
    b1_b4_boundaries_df[
        "board_sensor_index"
    ].notna().all()
)


# ---------------------------------------------------------
# Endpoint detector from an arbitrary start
# ---------------------------------------------------------

def board_lift_from_start(
    df,
    board_sensor_index,
    start_idx,
):
    """
    Detect object lift from a chosen start index.

    Processing:
        1. clean board saturation values
        2. select board_data_{trial_number}
        3. slice from chosen start
        4. zero-baseline at chosen start
        5. detect first crossing of 10% of maximum
        6. convert relative crossing back to absolute index

    Returns
    -------
    int or None
        Absolute recording index of detected object lift.
    """

    if start_idx is None:
        return None

    if pd.isna(start_idx):
        return None

    start_idx = int(
        start_idx
    )

    board_data = df[
        BOARD_COLUMNS
    ].to_numpy(
        dtype=float
    )

    board_data = replace_with_highest(
        board_data,
        target=8191.0,
    )

    board_data = replace_with_highest(
        board_data,
        target=8190.0,
    )


    signal = board_data[
        start_idx:-1,
        int(board_sensor_index),
    ]


    if len(signal) < 2:
        return None


    relative_signal = (
        signal
        - signal[0]
    )


    threshold = (
        0.1
        * relative_signal.max()
    )


    lift_relative = (
        first_change_index(
            relative_signal,
            threshold,
        )
    )


    if lift_relative is None:
        return None


    return int(
        start_idx
        + lift_relative
    )


# ---------------------------------------------------------
# Run endpoint comparison over all 640 B1-B4 recordings
# ---------------------------------------------------------

endpoint_records = []
endpoint_errors = []


for _, row in (
    b1_b4_boundaries_df
    .iterrows()
):

    source_path = Path(
        row["path"]
    )

    try:

        df = pd.read_csv(
            source_path
        )


        board_sensor_index = int(
            row[
                "board_sensor_index"
            ]
        )


        starts = {
            "recording0": 0,

            "hand_active":
                row[
                    "start_hand_active"
                ],

            "palm10":
                row[
                    "start_palm_10pct"
                ],

            "palm50":
                row[
                    "start_palm_50mm"
                ],
        }


        endpoints = {}


        for (
            start_name,
            start_idx,
        ) in starts.items():

            endpoints[
                start_name
            ] = board_lift_from_start(
                df=df,
                board_sensor_index=board_sensor_index,
                start_idx=start_idx,
            )


        endpoint_records.append(
            {
                "participant":
                    int(
                        row[
                            "participant"
                        ]
                    ),

                "block":
                    int(
                        row[
                            "block"
                        ]
                    ),

                "trial_number":
                    int(
                        row[
                            "trial_number"
                        ]
                    ),

                "rep_number":
                    int(
                        row[
                            "rep_number"
                        ]
                    ),

                "file":
                    row[
                        "file"
                    ],

                "path":
                    row[
                        "path"
                    ],

                "board_sensor_index":
                    board_sensor_index,

                "start_hand_active":
                    row[
                        "start_hand_active"
                    ],

                "start_palm_10pct":
                    row[
                        "start_palm_10pct"
                    ],

                "start_palm_50mm":
                    row[
                        "start_palm_50mm"
                    ],

                "end_recording0":
                    endpoints[
                        "recording0"
                    ],

                "end_hand_active":
                    endpoints[
                        "hand_active"
                    ],

                "end_palm10":
                    endpoints[
                        "palm10"
                    ],

                "end_palm50":
                    endpoints[
                        "palm50"
                    ],
            }
        )


    except Exception as exc:

        endpoint_errors.append(
            {
                "participant":
                    row[
                        "participant"
                    ],

                "block":
                    row[
                        "block"
                    ],

                "trial_number":
                    row[
                        "trial_number"
                    ],

                "rep_number":
                    row[
                        "rep_number"
                    ],

                "file":
                    row[
                        "file"
                    ],

                "path":
                    row[
                        "path"
                    ],

                "error":
                    str(
                        exc
                    ),
            }
        )


endpoint_compare_df = pd.DataFrame(
    endpoint_records
)

endpoint_errors_df = pd.DataFrame(
    endpoint_errors
)


# ---------------------------------------------------------
# Endpoint differences relative to hand_active definition
# ---------------------------------------------------------

endpoint_compare_df[
    "recording0_minus_hand_active_end"
] = (
    endpoint_compare_df[
        "end_recording0"
    ]
    - endpoint_compare_df[
        "end_hand_active"
    ]
)


endpoint_compare_df[
    "palm10_minus_hand_active_end"
] = (
    endpoint_compare_df[
        "end_palm10"
    ]
    - endpoint_compare_df[
        "end_hand_active"
    ]
)


endpoint_compare_df[
    "palm50_minus_hand_active_end"
] = (
    endpoint_compare_df[
        "end_palm50"
    ]
    - endpoint_compare_df[
        "end_hand_active"
    ]
)


# ---------------------------------------------------------
# Detection success
# ---------------------------------------------------------

print(
    "\nEndpoint records computed:",
    len(
        endpoint_compare_df
    ),
)

print(
    "Exceptions:",
    len(
        endpoint_errors_df
    ),
)


print(
    "\nEndpoint detection success:"
)

for col in [
    "end_recording0",
    "end_hand_active",
    "end_palm10",
    "end_palm50",
]:

    print(
        f"{col}:",
        endpoint_compare_df[
            col
        ].notna().sum(),
        "/",
        len(
            endpoint_compare_df
        ),
    )


# ---------------------------------------------------------
# Distribution of endpoint changes
# ---------------------------------------------------------

print(
    "\nEndpoint differences relative to hand_active, in samples:"
)

display(
    endpoint_compare_df[
        [
            "recording0_minus_hand_active_end",
            "palm10_minus_hand_active_end",
            "palm50_minus_hand_active_end",
        ]
    ]
    .describe()
    .T
)


# ---------------------------------------------------------
# Exact endpoint agreement
# ---------------------------------------------------------

print(
    "\nExact agreement with hand_active-based endpoint:"
)

for col in [
    "end_recording0",
    "end_palm10",
    "end_palm50",
]:

    valid = (
        endpoint_compare_df[
            col
        ].notna()
        & endpoint_compare_df[
            "end_hand_active"
        ].notna()
    )

    agreement = (
        endpoint_compare_df.loc[
            valid,
            col,
        ]
        ==
        endpoint_compare_df.loc[
            valid,
            "end_hand_active",
        ]
    )

    print(
        f"{col}:",
        agreement.sum(),
        "/",
        valid.sum(),
    )


# ---------------------------------------------------------
# Endpoint disagreement by trial number
# ---------------------------------------------------------

endpoint_by_trial = (
    endpoint_compare_df
    .groupby(
        "trial_number"
    )
    .agg(
        n=(
            "file",
            "size",
        ),

        median_palm10_end_difference=(
            "palm10_minus_hand_active_end",
            "median",
        ),

        max_abs_palm10_end_difference=(
            "palm10_minus_hand_active_end",
            lambda x:
            x.abs().max(),
        ),

        median_palm50_end_difference=(
            "palm50_minus_hand_active_end",
            "median",
        ),

        max_abs_palm50_end_difference=(
            "palm50_minus_hand_active_end",
            lambda x:
            x.abs().max(),
        ),

        median_recording0_end_difference=(
            "recording0_minus_hand_active_end",
            "median",
        ),

        max_abs_recording0_end_difference=(
            "recording0_minus_hand_active_end",
            lambda x:
            x.abs().max(),
        ),
    )
)


print(
    "\nEndpoint disagreement by trial:"
)

display(
    endpoint_by_trial
)


# ---------------------------------------------------------
# Largest palm50 versus hand_active endpoint disagreements
# ---------------------------------------------------------

largest_endpoint_disagreements = (
    endpoint_compare_df
    .assign(
        abs_palm50_end_difference=lambda x:
        x[
            "palm50_minus_hand_active_end"
        ].abs()
    )
    .sort_values(
        "abs_palm50_end_difference",
        ascending=False,
    )
    [
        [
            "participant",
            "block",
            "trial_number",
            "rep_number",
            "file",
            "start_hand_active",
            "start_palm_10pct",
            "start_palm_50mm",
            "end_recording0",
            "end_hand_active",
            "end_palm10",
            "end_palm50",
            "palm50_minus_hand_active_end",
            "abs_palm50_end_difference",
        ]
    ]
    .head(20)
)


print(
    "\nLargest palm50 vs hand_active endpoint disagreements:"
)

display(
    largest_endpoint_disagreements
)


# ---------------------------------------------------------
# Show exceptions if any
# ---------------------------------------------------------

if not endpoint_errors_df.empty:

    print(
        "\nEndpoint extraction errors:"
    )

    display(
        endpoint_errors_df
    )

Resolved paths: 640 / 640
Resolved board sensor indices: 640 / 640

Endpoint records computed: 640
Exceptions: 0

Endpoint detection success:
end_recording0: 640 / 640
end_hand_active: 640 / 640
end_palm10: 640 / 640
end_palm50: 640 / 640

Endpoint differences relative to hand_active, in samples:


,count,mean,std,min,25%,50%,75%,max
recording0_minus_hand_active_end,640.0,-0.101562,7.284988,-150.0,0.0,0.0,0.0,54.0
palm10_minus_hand_active_end,640.0,-0.012500,3.842335,-18.0,0.0,0.0,0.0,42.0
palm50_minus_hand_active_end,640.0,0.076563,4.555560,-49.0,0.0,0.0,0.0,52.0



Exact agreement with hand_active-based endpoint:
end_recording0: 515 / 640
end_palm10: 501 / 640
end_palm50: 508 / 640

Endpoint disagreement by trial:


,n,median_palm10_end_difference,max_abs_palm10_end_difference,median_palm50_end_difference,max_abs_palm50_end_difference,median_recording0_end_difference,max_abs_recording0_end_difference
trial_number,,,,,,,
0,80,0.0,12,0.0,10,0.0,14
1,80,0.0,42,0.0,52,0.0,150
2,80,0.0,10,0.0,10,0.0,4
3,80,0.0,19,0.0,18,0.0,18
4,80,0.0,32,0.0,18,0.0,12
5,80,0.0,18,0.0,18,0.0,18
6,80,0.0,33,0.0,33,0.0,33
7,80,0.0,8,0.0,10,0.0,10



Largest palm50 vs hand_active endpoint disagreements:


,participant,block,trial_number,rep_number,file,start_hand_active,start_palm_10pct,start_palm_50mm,end_recording0,end_hand_active,end_palm10,end_palm50,palm50_minus_hand_active_end,abs_palm50_end_difference
293,3,2,1,1,B2_trial1_rep1_attempt0.csv,232,215,223,293,239,221,291,52,52
263,3,1,1,3,B1_trial1_rep3_attempt0.csv,223,207,215,282,282,282,233,-49,49
452,4,3,1,0,B3_trial1_rep0_attempt0.csv,213,199,205,281,239,281,281,42,42
88,1,3,6,0,B3_trial6_rep0_attempt0.csv,218,205,213,304,271,304,304,33,33
327,3,3,1,3,B3_trial1_rep3_attempt0.csv,216,206,212,269,245,245,269,24,24
260,3,1,1,0,B1_trial1_rep0_attempt0.csv,225,213,217,280,260,280,280,20,20
405,4,1,5,1,B1_trial5_rep1_attempt0.csv,223,211,215,295,277,295,295,18,18
175,2,2,3,3,B2_trial3_rep3_attempt0.csv,361,260,270,345,363,345,345,-18,18
371,3,4,4,3,B4_trial4_rep3_attempt0.csv,231,220,224,277,265,265,283,18,18
140,2,1,3,0,B1_trial3_rep0_attempt0.csv,185,167,174,277,261,277,277,16,16


### B1-B4 boundary reconstruction: interim findings

The original source code contains several related definitions of movement timing in B1-B4. Before choosing a CTM crop, these definitions were compared directly across all 640 retained B1-B4 recordings.

All candidate events could be detected in all 640 recordings:

- first `hand_active == 1`
- palm displacement crossing 10% of its relative maximum
- palm displacement crossing 50 mm
- target-board displacement crossing 10% of its relative maximum

The onset definitions are not equivalent. The 50-mm palm crossing generally precedes the first `hand_active` sample, and the discrepancy is especially pronounced for trial 3. In trial 3, 20 of 80 recordings differ by more than 300 ms, 10 differ by more than 500 ms, and 12 produce a `hand_active`-based movement segment shorter than 20 samples.

The object-lift endpoint is substantially more robust. Recomputing the 10%-of-maximum board crossing after zero-baselining the target-board signal from four different starting points produced a median endpoint difference of zero samples for every comparison and every trial number.

Therefore, the principal unresolved methodological issue is the definition of movement onset rather than the definition of object lift.

The repository uses `hand_active` as an existing metadata signal but does not define how that signal was originally generated. Consequently, the final B1-B4 CTM movement crop should not yet be fixed until the provenance of `hand_active` and its relationship to the rest-position sensor criterion have been clarified.

### B1-B4 extraction and unified loader validation

The B1-B4 movement extraction was reconstructed from the original source implementation used for the reach-to-grasp trajectories.

For B1-B4:

- movement start is the first sample where `hand_active == 1`
- the relevant object sensor is `board_data_{trial_number}`
- the board signal is zero-baselined at movement start
- object lift is the first crossing of 10% of the maximum relative board displacement
- the extracted trajectory ends at object lift, without the additional 50 samples used for B0

This intentionally differs from B0, where movement onset is detected from glove speed, the endpoint uses 20% of `board_data_3`, and the exported trajectory retains 50 additional samples after the detected endpoint.

The B1-B4 implementation successfully extracted all 640 retained recordings. All extracted trajectories contained 15 fingertip channels and all resampled trajectories were finite.

The final unified loader produced:

$$
X \in \mathbb{R}^{958 \times 15 \times 100}
$$

with 958 labels and 958 corresponding metadata records.

The retained recordings are distributed as:

- B0: 318
- B1: 160
- B2: 160
- B3: 160
- B4: 160

The final class counts are:

- class 0: 120
- class 1: 120
- class 2: 120
- class 3: 119
- class 4: 119
- class 5: 120
- class 6: 120
- class 7: 120

All final CTM input trajectories are finite and have shape `(15, 100)`.

A small number of B1-B4 recordings, predominantly involving trial 3, have unusually short `hand_active`-based intervals. These are retained because they arise from the reproduced source extraction rule rather than from a newly introduced preprocessing criterion. Their influence can be examined separately if necessary.

# Appendix A. Historical Development and Rejected Assumptions

This appendix preserves the main exploratory steps that led to the current preprocessing pipeline.

These analyses are no longer part of the active loader specification, but they are retained because they explain why several apparently reasonable preprocessing assumptions were rejected.

## A.1 First-pass loader

The first version of the reach-to-grasp loader used a simplified interpretation of the processed recordings.

It approximately followed:

    read trial
        |
        v
    use hand_active as movement onset
        |
        v
    identify an object-board signal heuristically
        |
        v
    determine an endpoint
        |
        v
    crop fingertip trajectory
        |
        v
    resample to fixed length

A single-trial test produced the expected tensor shape, but participant-level evaluation revealed substantial problems.

For participant 1:

    physical processed files = 193
    successfully loaded = 183
    skipped failed recordings = 1
    loader errors = 9

The resulting class distribution was also strongly inconsistent with the intended balanced eight-class experiment.

This showed that producing correctly shaped arrays was not sufficient evidence that the dataset semantics had been interpreted correctly.

The loader was therefore decomposed into smaller metadata, selection, mapping, and boundary-detection stages.

## A.2 Investigating hand_active as movement onset

The first-pass loader used:

    hand_active == 1

as the movement-start criterion.

Because the dataset also contains a rest-position ToF signal, an exploratory alternative onset was defined using `palm_data`.

For participant 1, the first crossing of approximately 50 mm above the resting palm position was compared with the first `hand_active` sample.

Across all 192 successful participant-1 recordings, the palm-derived event occurred earlier than `hand_active`.

The observed delay had approximately:

    mean = 138.3 ms
    median = 112 ms
    minimum = 14 ms
    maximum = 722 ms

For the original validation recording:

    B0_trial0_rep1_attempt0.csv

the diagnostic timings were:

    palm +50 mm onset index = 229
    palm onset time = 3598 ms

    hand_active index = 239
    hand_active time = 3754 ms

    difference = 156 ms

This analysis demonstrated that `hand_active` should not automatically be interpreted as physical movement onset.

However, the palm +50 mm rule was itself only a diagnostic hypothesis.

Inspection of the original source code later showed that the B0 movement-export pipeline determines movement start from the glove-derived speed profile.

The palm analysis is therefore retained as historical evidence that multiple onset definitions exist, not as the final B0 preprocessing rule.

## A.3 Discovering changes within target_object

Early loader versions treated `target_object` as if it were constant throughout each recording.

Inspection of the processed CSV files showed that this is not always true.

Some recordings contain sequences such as:

    [3, 0]

or other changes during the recording.

This explained why reading only the first row could produce misleading metadata.

The current loader instead records:

    target_object_final
    target_values_seen
    target_changed_within_recording

and uses the final valid non-missing target where required.

A diagnostic count found 139 recordings containing more than one distinct non-missing target value.

An earlier count of 146 also included transitions from missing values to populated values.

The two counts therefore correspond to different definitions.

## A.4 Rejected direct target-to-board mapping

An early hypothesis was that object identity might directly determine the physical board channel:

    target_object -> board_data_{target_object}

This was tested empirically on 192 successful participant-1 recordings.

The board channel associated with `target_object` was compared with the channel showing the strongest response.

Exact agreement occurred in only:

$$
\frac{88}{192}
\approx 45.8\%.
$$

The observed rank distribution was:

    rank 1 -> 88
    rank 2 -> 14
    rank 3 -> 17
    rank 4 -> 14
    rank 5 -> 18
    rank 6 -> 15
    rank 7 -> 13
    rank 8 -> 13

This was far too weak to support a direct object-to-board mapping.

The hypothesis was therefore rejected.

Subsequent source-code reconstruction and experimental clarification established the block-specific rules now used in the main preprocessing pipeline:

    B0:
        class label = trial_number
        board sensor = board_data_3

    B1-B4:
        class label = corrected target_object
        board sensor = board_data_{trial_number}

## A.5 Why this history is retained

These exploratory steps illustrate an important principle of the project:

    plausible interpretation
          |
          v
    empirical diagnostic
          |
          v
    compare with source code
          |
          v
    retain or reject assumption

The current loader should therefore be understood as the result of progressively resolving dataset semantics rather than as a direct translation of the first preprocessing idea.